# 🔗 Notebook 16 — Trimodal Fusion (Text+Audio+Video)

**Thesis:** Explainable AI for Multimodal Content Moderation  
**Author:** Md. Safayat Bin Nasir (2004099, CSE CUET)

---

### 🎯 এই notebook
**{BanglaBERT, XLM-R} × {HuBERT, wav2vec2} × {CNN, ViT, ViViT} × {early, late, crossattn, gated}** = **48 combinations**

### 📋 Add Input করো (Your Work → Notebooks)
| Notebook | Embeddings |
|---|---|
| banglabert-text | emb_BanglaBERT_*.npy |
| xlm-r-text | emb_XLM-R_*.npy |
| hubert-audio | emb_HuBERT_*.npy |
| wav2vec2-audio | emb_wav2vec2_*.npy |
| cnn-video | emb_CNN_*.npy |
| vit-video | emb_ViT_*.npy |
| vivit-video | emb_ViViT_*.npy |

### 📊 Bimodal Best (reference)
| Pair | Best | F1 |
|---|---|---|
| Text+Video | BanglaBERT+ViT_crossattn | 0.8501 |
| Text+Audio | BanglaBERT+HuBERT_gated | 0.8434 |
| Audio+Video | HuBERT+CNN_late | 0.7664 |


## 1️⃣ Install

In [1]:
!pip install -q torch scikit-learn numpy pandas
print("✅ Install done")

✅ Install done


## 2️⃣ Imports & Device

In [2]:
import os, json, glob, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_recall_fscore_support,
                             classification_report, confusion_matrix)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device : {DEVICE}")
print(f"✅ PyTorch: {torch.__version__}")

✅ Device : cuda
✅ PyTorch: 2.10.0+cu128


## 3️⃣ Configuration

⚠️ **Add Input করার পরে Cell 4 চালিয়ে actual path দেখো, তারপর EMB_SEARCH_DIRS update করো।**

In [3]:
# ── Output path ───────────────────────────────
OUT_DIR = '/kaggle/working'

# ── Embedding search paths ────────────────────
EMB_SEARCH_DIRS = [
    '/kaggle/working',
    # Text
    '/kaggle/input/notebooks/mdsafayatbinnasir/banglabert-text',
    '/kaggle/input/notebooks/mdsafayatbinnasir/xlm-r-text',
    # Audio
    '/kaggle/input/notebooks/mdsafayatbinnasir/hubert-audio',
    '/kaggle/input/notebooks/mdsafayatbinnasir/wav2vec2-audio',
    # Video (path check করে বসাও)
    '/kaggle/input/notebooks/mdsafayatbinnasir/09-cnn-video',
    '/kaggle/input/notebooks/mdsafayatbinnasir/07-vit-video',
    '/kaggle/input/notebooks/mdsafayatbinnasir/10-vivit-video',
    '/kaggle/input/notebooks/mdsafayatbinnasir/whisper-audio',
]

# ── Run these combinations ────────────────────
# (TEXT, AUDIO, VIDEO, FUSION)
COMBINATIONS = []
for text in ['BanglaBERT', 'XLM-R']:
    for audio in ['HuBERT', 'wav2vec2','Whisper']:
        for video in ['CNN', 'ViT', 'ViViT']:
            for fusion in ['early', 'late', 'crossattn', 'gated']:
                COMBINATIONS.append((text, audio, video, fusion))

# ── Hyperparameters ───────────────────────────
SEED         = 42
TEST_SIZE    = 0.15
VAL_SIZE     = 0.15
BATCH_SIZE   = 32
EPOCHS       = 20
LR           = 1e-3
WEIGHT_DECAY = 0.01
PATIENCE     = 5

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"✅ {len(COMBINATIONS)} combinations to run:")
for i, (t, a, v, f) in enumerate(COMBINATIONS, 1):
    print(f"   {i:2d}. {t} + {a} + {v} [{f}]")

✅ 72 combinations to run:
    1. BanglaBERT + HuBERT + CNN [early]
    2. BanglaBERT + HuBERT + CNN [late]
    3. BanglaBERT + HuBERT + CNN [crossattn]
    4. BanglaBERT + HuBERT + CNN [gated]
    5. BanglaBERT + HuBERT + ViT [early]
    6. BanglaBERT + HuBERT + ViT [late]
    7. BanglaBERT + HuBERT + ViT [crossattn]
    8. BanglaBERT + HuBERT + ViT [gated]
    9. BanglaBERT + HuBERT + ViViT [early]
   10. BanglaBERT + HuBERT + ViViT [late]
   11. BanglaBERT + HuBERT + ViViT [crossattn]
   12. BanglaBERT + HuBERT + ViViT [gated]
   13. BanglaBERT + wav2vec2 + CNN [early]
   14. BanglaBERT + wav2vec2 + CNN [late]
   15. BanglaBERT + wav2vec2 + CNN [crossattn]
   16. BanglaBERT + wav2vec2 + CNN [gated]
   17. BanglaBERT + wav2vec2 + ViT [early]
   18. BanglaBERT + wav2vec2 + ViT [late]
   19. BanglaBERT + wav2vec2 + ViT [crossattn]
   20. BanglaBERT + wav2vec2 + ViT [gated]
   21. BanglaBERT + wav2vec2 + ViViT [early]
   22. BanglaBERT + wav2vec2 + ViViT [late]
   23. BanglaBERT + wav2ve

## 4️⃣ Path Check

In [4]:
print("=== All .npy files found ===")
found = glob.glob('/kaggle/input/**/*.npy', recursive=True)
if found:
    for f in sorted(found):
        print(f)
else:
    print("❌ No .npy files found! Add Input করো।")
print(f"\nTotal: {len(found)} files")

=== All .npy files found ===
/kaggle/input/notebooks/mdsafayatbinnasir/07-vit-video/emb_ViT_test.npy
/kaggle/input/notebooks/mdsafayatbinnasir/07-vit-video/emb_ViT_train.npy
/kaggle/input/notebooks/mdsafayatbinnasir/07-vit-video/emb_ViT_val.npy
/kaggle/input/notebooks/mdsafayatbinnasir/09-cnn-video/emb_CNN_test.npy
/kaggle/input/notebooks/mdsafayatbinnasir/09-cnn-video/emb_CNN_train.npy
/kaggle/input/notebooks/mdsafayatbinnasir/09-cnn-video/emb_CNN_val.npy
/kaggle/input/notebooks/mdsafayatbinnasir/10-vivit-video/emb_ViViT_test.npy
/kaggle/input/notebooks/mdsafayatbinnasir/10-vivit-video/emb_ViViT_train.npy
/kaggle/input/notebooks/mdsafayatbinnasir/10-vivit-video/emb_ViViT_val.npy
/kaggle/input/notebooks/mdsafayatbinnasir/banglabert-text/emb_BanglaBERT_test.npy
/kaggle/input/notebooks/mdsafayatbinnasir/banglabert-text/emb_BanglaBERT_train.npy
/kaggle/input/notebooks/mdsafayatbinnasir/banglabert-text/emb_BanglaBERT_val.npy
/kaggle/input/notebooks/mdsafayatbinnasir/hubert-audio/emb_HuBERT

## 5️⃣ Load Labels & Split (once)

In [5]:
BASE     = '/kaggle/input/datasets/mdsafayatbinnasir/bangla-codemixed-toxic-dataset'
CSV_PATH = f'{BASE}/Toxic-NonToxic_Content_Dataset - Sheet4.csv'

df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]
df = df.dropna(subset=['Toxic']).copy()
df['label'] = df['Toxic'].astype(int)

train_val, test = train_test_split(df, test_size=TEST_SIZE,
                                   stratify=df['label'], random_state=SEED)
val_ratio = VAL_SIZE / (1 - TEST_SIZE)
train, val = train_test_split(train_val, test_size=val_ratio,
                               stratify=train_val['label'], random_state=SEED)

train = train.reset_index(drop=True)
val   = val.reset_index(drop=True)
test  = test.reset_index(drop=True)

y_train = torch.LongTensor(train['label'].values)
y_val   = torch.LongTensor(val['label'].values)
y_test  = torch.LongTensor(test['label'].values)

n  = np.bincount(train['label'].values, minlength=2)
cw = torch.tensor(n.sum() / (2*np.maximum(n,1)), dtype=torch.float).to(DEVICE)

print(f"✅ Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")
print(f"   Class weights: non-toxic={cw[0]:.3f}, toxic={cw[1]:.3f}")

✅ Train: 775 | Val: 167 | Test: 167
   Class weights: non-toxic=0.991, toxic=1.009


## 6️⃣ Embedding Loader (cache for speed)

In [6]:
_emb_cache = {}

def load_emb(tag, split):
    key = f"{tag}_{split}"
    if key not in _emb_cache:
        fname = f'emb_{tag}_{split}.npy'
        found = None
        for d in EMB_SEARCH_DIRS:
            path = os.path.join(d, fname)
            if os.path.exists(path):
                found = path
                break
        if found is None:
            raise FileNotFoundError(
                f"❌ Not found: {fname}\n"
                f"   Add Input করো এবং EMB_SEARCH_DIRS update করো!")
        _emb_cache[key] = np.load(found)
        print(f"  Loaded: {found}")
    return _emb_cache[key]

def make_loader(Xt, Xa, Xv, y, shuffle):
    ds = TensorDataset(torch.FloatTensor(Xt),
                       torch.FloatTensor(Xa),
                       torch.FloatTensor(Xv), y)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

print("✅ Embedding loader ready (with cache)")

✅ Embedding loader ready (with cache)


## 7️⃣ Trimodal Fusion Models

4 strategies — সবগুলোতেই **3 modality** (Text, Audio, Video):
- **Early:** Concatenate all 3 → MLP
- **Late:** Separate head per modality → average logits
- **CrossAttn:** Pairwise cross-modal attention (T↔A, T↔V, A↔V) → concat → MLP
- **Gated:** Learnable gate per modality → weighted sum → MLP


In [7]:
# ── Early Fusion (3 modalities) ──────────────
class EarlyFusion3(nn.Module):
    def __init__(self, dt, da, dv, nc=2, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(dt+da+dv),
            nn.Linear(dt+da+dv, 512), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(512, 256),       nn.ReLU(), nn.Dropout(drop),
            nn.Linear(256, nc))
    def forward(self, t, a, v):
        return self.net(torch.cat([t, a, v], dim=-1))

# ── Late Fusion (3 modalities) ────────────────
class LateFusion3(nn.Module):
    def __init__(self, dt, da, dv, nc=2, drop=0.3):
        super().__init__()
        self.ht = nn.Sequential(nn.LayerNorm(dt),
                                nn.Linear(dt,256), nn.ReLU(), nn.Dropout(drop),
                                nn.Linear(256,nc))
        self.ha = nn.Sequential(nn.LayerNorm(da),
                                nn.Linear(da,256), nn.ReLU(), nn.Dropout(drop),
                                nn.Linear(256,nc))
        self.hv = nn.Sequential(nn.LayerNorm(dv),
                                nn.Linear(dv,256), nn.ReLU(), nn.Dropout(drop),
                                nn.Linear(256,nc))
    def forward(self, t, a, v):
        return (self.ht(t) + self.ha(a) + self.hv(v)) / 3

# ── Cross-Attention Fusion (3 modalities) ─────
class CrossAttnFusion3(nn.Module):
    def __init__(self, dt, da, dv, nheads=4, nc=2, drop=0.3):
        super().__init__()
        pd = 256
        # projections
        self.pt = nn.Linear(dt, pd)
        self.pa = nn.Linear(da, pd)
        self.pv = nn.Linear(dv, pd)
        # T ↔ A
        self.ta = nn.MultiheadAttention(pd, nheads, dropout=drop, batch_first=True)
        self.at = nn.MultiheadAttention(pd, nheads, dropout=drop, batch_first=True)
        # T ↔ V
        self.tv = nn.MultiheadAttention(pd, nheads, dropout=drop, batch_first=True)
        self.vt = nn.MultiheadAttention(pd, nheads, dropout=drop, batch_first=True)
        # A ↔ V
        self.av = nn.MultiheadAttention(pd, nheads, dropout=drop, batch_first=True)
        self.va = nn.MultiheadAttention(pd, nheads, dropout=drop, batch_first=True)
        # norms
        self.nt = nn.LayerNorm(pd); self.na = nn.LayerNorm(pd); self.nv = nn.LayerNorm(pd)
        # head (3 × pd → nc)
        self.head = nn.Sequential(
            nn.Linear(pd*3, 512), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(512, 256),  nn.ReLU(), nn.Dropout(drop),
            nn.Linear(256, nc))
    def forward(self, t, a, v):
        pt = self.pt(t).unsqueeze(1)  # (B,1,pd)
        pa = self.pa(a).unsqueeze(1)
        pv = self.pv(v).unsqueeze(1)
        # T attends A, A attends T
        ta,_ = self.ta(pt, pa, pa); at_,_ = self.at(pa, pt, pt)
        # T attends V, V attends T
        tv,_ = self.tv(pt, pv, pv); vt_,_ = self.vt(pv, pt, pt)
        # A attends V, V attends A
        av,_ = self.av(pa, pv, pv); va_,_ = self.va(pv, pa, pa)
        # residual + norm
        ft = self.nt(pt.squeeze(1) + ta.squeeze(1) + tv.squeeze(1))
        fa = self.na(pa.squeeze(1) + at_.squeeze(1) + av.squeeze(1))
        fv = self.nv(pv.squeeze(1) + vt_.squeeze(1) + va_.squeeze(1))
        return self.head(torch.cat([ft, fa, fv], dim=-1))

# ── Gated Fusion (3 modalities) ───────────────
class GatedFusion3(nn.Module):
    def __init__(self, dt, da, dv, nc=2, drop=0.3):
        super().__init__()
        pd = 256
        self.pt = nn.Sequential(nn.LayerNorm(dt), nn.Linear(dt,pd), nn.ReLU())
        self.pa = nn.Sequential(nn.LayerNorm(da), nn.Linear(da,pd), nn.ReLU())
        self.pv = nn.Sequential(nn.LayerNorm(dv), nn.Linear(dv,pd), nn.ReLU())
        self.gate = nn.Sequential(nn.Linear(pd*3, pd*3), nn.Sigmoid())
        self.head = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(pd*3, 256), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(256, nc))
    def forward(self, t, a, v):
        pt = self.pt(t); pa = self.pa(a); pv = self.pv(v)
        c  = torch.cat([pt, pa, pv], dim=-1)
        return self.head(self.gate(c) * c)

def build_model(fusion, dt, da, dv):
    if fusion == 'early':     return EarlyFusion3(dt, da, dv)
    if fusion == 'late':      return LateFusion3(dt, da, dv)
    if fusion == 'crossattn': return CrossAttnFusion3(dt, da, dv)
    if fusion == 'gated':     return GatedFusion3(dt, da, dv)
    raise ValueError(f"Unknown fusion: {fusion}")

print("✅ All trimodal fusion models defined")

✅ All trimodal fusion models defined


## 8️⃣ Train & Evaluate Function

In [8]:
def run_epoch(model, loader, train_mode, criterion, optimizer):
    model.train() if train_mode else model.eval()
    total_loss, preds, labels = 0, [], []
    torch.set_grad_enabled(train_mode)
    for t, a, v, y in loader:
        t, a, v, y = t.to(DEVICE), a.to(DEVICE), v.to(DEVICE), y.to(DEVICE)
        if train_mode: optimizer.zero_grad()
        out  = model(t, a, v)
        loss = criterion(out, y)
        if train_mode:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total_loss += loss.item()
        preds.extend(out.argmax(1).cpu().numpy())
        labels.extend(y.cpu().numpy())
    torch.set_grad_enabled(True)
    return total_loss/len(loader), f1_score(labels, preds, average='macro', zero_division=0)

def train_and_eval(mod_t, mod_a, mod_v, fusion, dt, da, dv):
    tag = f"{mod_t}+{mod_a}+{mod_v}_{fusion}"
    print(f"\n{'='*60}")
    print(f"  Running: {tag}")
    print(f"{'='*60}")

    Xt_tr = load_emb(mod_t, 'train'); Xa_tr = load_emb(mod_a, 'train'); Xv_tr = load_emb(mod_v, 'train')
    Xt_vl = load_emb(mod_t, 'val');   Xa_vl = load_emb(mod_a, 'val');   Xv_vl = load_emb(mod_v, 'val')
    Xt_te = load_emb(mod_t, 'test');  Xa_te = load_emb(mod_a, 'test');  Xv_te = load_emb(mod_v, 'test')

    train_loader = make_loader(Xt_tr, Xa_tr, Xv_tr, y_train, True)
    val_loader   = make_loader(Xt_vl, Xa_vl, Xv_vl, y_val,   False)
    test_loader  = make_loader(Xt_te, Xa_te, Xv_te, y_test,  False)

    model     = build_model(fusion, dt, da, dv).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=cw)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_f1, patience_cnt, best_state = 0, 0, None
    for ep in range(EPOCHS):
        tr_loss, tr_f1 = run_epoch(model, train_loader, True,  criterion, optimizer)
        vl_loss, vl_f1 = run_epoch(model, val_loader,   False, criterion, optimizer)
        print(f"  Ep{ep+1:02d} | tr_loss={tr_loss:.4f} f1={tr_f1:.4f} "
              f"| val_loss={vl_loss:.4f} f1={vl_f1:.4f}", end="")
        if vl_f1 > best_f1:
            best_f1 = vl_f1
            best_state = {k: v.clone() for k,v in model.state_dict().items()}
            patience_cnt = 0
            print(f" ✓")
        else:
            patience_cnt += 1
            print()
            if patience_cnt >= PATIENCE:
                print(f"  ⏹ early stop at epoch {ep+1}")
                break

    model.load_state_dict(best_state)
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for t, a, v, y in test_loader:
            out = model(t.to(DEVICE), a.to(DEVICE), v.to(DEVICE))
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(y.numpy())

    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average='macro', zero_division=0)
    tf1 = f1_score(labels, preds, labels=[1], average='macro', zero_division=0)
    p, r, _, _ = precision_recall_fscore_support(labels, preds,
                                                  average='macro', zero_division=0)

    result = {
        'tag': tag, 'text': mod_t, 'audio': mod_a, 'video': mod_v,
        'fusion': fusion, 'accuracy': round(acc,4), 'macro_f1': round(f1,4),
        'toxic_f1': round(tf1,4), 'precision': round(p,4), 'recall': round(r,4)
    }

    with open(f'{OUT_DIR}/result_{tag}.json', 'w') as fp:
        json.dump(result, fp, indent=2)

    print(f"\n  ✅ {tag}: Macro-F1={f1:.4f} | Toxic-F1={tf1:.4f} | Acc={acc:.4f}")
    return result

print("✅ Train & evaluate function ready")

✅ Train & evaluate function ready


## 9️⃣ Run All 48 Combinations

In [9]:
all_results = []

for mod_t, mod_a, mod_v, fusion in COMBINATIONS:
    dt = load_emb(mod_t, 'train').shape[1]
    da = load_emb(mod_a, 'train').shape[1]
    dv = load_emb(mod_v, 'train').shape[1]
    res = train_and_eval(mod_t, mod_a, mod_v, fusion, dt, da, dv)
    all_results.append(res)

print("\n" + "="*60)
print("  ALL RESULTS SUMMARY (Trimodal: Text+Audio+Video)")
print("="*60)
print(f"  {'Combination':<52} {'F1':>6} {'Acc':>6}")
print(f"  {'-'*66}")
for r in sorted(all_results, key=lambda x: x['macro_f1'], reverse=True):
    print(f"  {r['tag']:<52} {r['macro_f1']:>6.4f} {r['accuracy']:>6.4f}")

best = max(all_results, key=lambda x: x['macro_f1'])
print(f"\n  🏆 Best Trimodal: {best['tag']}")
print(f"      Macro-F1  = {best['macro_f1']:.4f}")
print(f"      Toxic-F1  = {best['toxic_f1']:.4f}")
print(f"      Accuracy  = {best['accuracy']:.4f}")

print(f"\n  ── Progressive improvement ──────────────────")
print(f"  Unimodal  BanglaBERT:                   0.8243")
print(f"  Bimodal   BanglaBERT+ViT_crossattn:     0.8501")
print(f"  Trimodal  {best['tag']}: {best['macro_f1']:.4f}")

  Loaded: /kaggle/input/notebooks/mdsafayatbinnasir/banglabert-text/emb_BanglaBERT_train.npy
  Loaded: /kaggle/input/notebooks/mdsafayatbinnasir/hubert-audio/emb_HuBERT_train.npy
  Loaded: /kaggle/input/notebooks/mdsafayatbinnasir/09-cnn-video/emb_CNN_train.npy

  Running: BanglaBERT+HuBERT+CNN_early
  Loaded: /kaggle/input/notebooks/mdsafayatbinnasir/banglabert-text/emb_BanglaBERT_val.npy
  Loaded: /kaggle/input/notebooks/mdsafayatbinnasir/hubert-audio/emb_HuBERT_val.npy
  Loaded: /kaggle/input/notebooks/mdsafayatbinnasir/09-cnn-video/emb_CNN_val.npy
  Loaded: /kaggle/input/notebooks/mdsafayatbinnasir/banglabert-text/emb_BanglaBERT_test.npy
  Loaded: /kaggle/input/notebooks/mdsafayatbinnasir/hubert-audio/emb_HuBERT_test.npy
  Loaded: /kaggle/input/notebooks/mdsafayatbinnasir/09-cnn-video/emb_CNN_test.npy
  Ep01 | tr_loss=0.3361 f1=0.9213 | val_loss=1.0338 f1=0.8443 ✓
  Ep02 | tr_loss=0.1786 f1=0.9664 | val_loss=0.4869 f1=0.8443
  Ep03 | tr_loss=0.1345 f1=0.9664 | val_loss=0.5555 f1=0.